# 04 — Gold Feature Engineering | NYC Taxi Demand Prediction

**Microsoft Fabric · PySpark · Delta Lake · 2025 · 5 zones de Manhattan**

## Objectif
Produire une table Gold **une ligne par zone de taxi × heure locale**, avec :
- `pickup_count` (cible = nombre de trajets, y compris 0) ;
- les conditions météorologiques horaires issues de `silver.weather_hourly` ;
- des caractéristiques calendaires connues à l'avance (heure, jour, mois, week-end, composantes cycliques).

**Sources :** `silver.taxi_trips` et `silver.weather_hourly`. Le notebook **ne modifie jamais Silver ni Bronze**.

**Important pour la prédiction autonome :** aucune variable `lag_1`, `lag_24`, moyenne mobile ou demande réelle récente dans ce dataset de base : faute de flux NYC Taxi temps réel, ces variables ne seraient pas disponibles au moment de l'inférence. Une version de recherche avec lags peut être développée séparément et évaluée dans un cadre strictement historique.

**Attention aux fuseaux :** les deux sources sont représentées comme heures locales de New York, sans décalage UTC. Ce notebook utilise une grille de 8 760 heures civiles nominales en 2025. Les changements d'heure (DST : heure inexistante au printemps et heure répétée en automne) ne peuvent pas être parfaitement modélisés avec ces timestamps seuls. C'est une **limite connue** de cette première version ; pour une version rigoureuse, réingérer des horodatages avec offset/UTC et adapter la grille.

## 1 — Configuration et lecture des tables Silver
Attache `lh_nyc_taxi` comme **Lakehouse par défaut** dans ce notebook Microsoft Fabric. La table Gold est écrite avec `overwrite` uniquement à la toute fin après validation.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
import math

YEAR = 2025
ZONE_IDS = [230, 161, 237, 186, 87]
EXPECTED_HOURS = 365 * 24
EXPECTED_ROWS = len(ZONE_IDS) * EXPECTED_HOURS
TAXI_TABLE = "silver.taxi_trips"
WEATHER_TABLE = "silver.weather_hourly"
GOLD_TABLE = "gold.taxi_demand_features"

print("Spark :", spark.version)
print("Gold cible :", GOLD_TABLE)
print("Grille nominale attendue :", EXPECTED_ROWS, "lignes")

In [ ]:
df_taxi = spark.table(TAXI_TABLE)
df_weather = spark.table(WEATHER_TABLE)

print("=== TAXI ===")
df_taxi.printSchema()
print("=== METEO ===")
df_weather.printSchema()

required_taxi = {"PULocationID", "tpep_pickup_datetime"}
required_weather = {"taxi_zone_id", "weather_hour_local"}
assert required_taxi.issubset(df_taxi.columns), required_taxi - set(df_taxi.columns)
assert required_weather.issubset(df_weather.columns), required_weather - set(df_weather.columns)
assert isinstance(df_taxi.schema["tpep_pickup_datetime"].dataType, T.TimestampNTZType), "Date Taxi attendue en TIMESTAMP_NTZ"
assert isinstance(df_weather.schema["weather_hour_local"].dataType, T.TimestampNTZType), "Date météo attendue en TIMESTAMP_NTZ"

n_taxi = df_taxi.count()
n_weather = df_weather.count()
print(f"Silver Taxi : {n_taxi:,} trajets")
print(f"Silver météo : {n_weather:,} observations")

## 2 — Contrôles de qualité Silver avant toute jointure
Nous contrôlons la période, les cinq zones et **l'unicité `(taxi_zone_id, weather_hour_local)`**. Si un contrôle échoue, corriger Silver avant Gold.

In [ ]:
print("=== Comptage Taxi par zone ===")
display(df_taxi.groupBy("PULocationID").count().orderBy("PULocationID"))
print("=== Comptage météo par zone ===")
display(df_weather.groupBy("taxi_zone_id").agg(
    F.count("*").alias("lignes"),
    F.countDistinct("weather_hour_local").alias("heures_distinctes"),
    F.min("weather_hour_local").alias("debut"),
    F.max("weather_hour_local").alias("fin")
).orderBy("taxi_zone_id"))

invalid_taxi = df_taxi.filter(
    (~F.col("PULocationID").isin(ZONE_IDS)) |
    (F.col("tpep_pickup_datetime") < F.lit("2025-01-01")) |
    (F.col("tpep_pickup_datetime") >= F.lit("2026-01-01")) |
    F.col("PULocationID").isNull() |
    F.col("tpep_pickup_datetime").isNull()
).limit(1).count()
invalid_weather = df_weather.filter(
    (~F.col("taxi_zone_id").isin(ZONE_IDS)) |
    (F.col("weather_hour_local") < F.lit("2025-01-01")) |
    (F.col("weather_hour_local") >= F.lit("2026-01-01")) |
    F.col("taxi_zone_id").isNull() |
    F.col("weather_hour_local").isNull()
).limit(1).count()
weather_duplicate_keys = (df_weather.groupBy("taxi_zone_id", "weather_hour_local")
    .count().filter(F.col("count") > 1).limit(1).count())

assert invalid_taxi == 0, "Taxi : date/zone hors périmètre ou clé NULL"
assert invalid_weather == 0, "Météo : date/zone hors périmètre ou clé NULL"
assert weather_duplicate_keys == 0, "Météo : clés zone + heure dupliquées"
assert n_weather == EXPECTED_ROWS, f"Météo : {n_weather} lignes vs {EXPECTED_ROWS} nominales"
print("Contrôles de base Silver OK")

## 3 — Agréger les trajets par zone et par heure
Chaque ligne de Taxi Silver correspond à un trajet. On compte les **prises en charge (pickups)** selon `PULocationID` et l'heure tronquée. Il n'y a **pas** de jointure avec la météo à ce stade.

In [ ]:
df_pickups = (
    df_taxi
    .select(
        F.col("PULocationID").cast("int").alias("taxi_zone_id"),
        F.date_trunc("hour", F.col("tpep_pickup_datetime"))
            .cast("timestamp_ntz").alias("hour_local")
    )
    .groupBy("taxi_zone_id", "hour_local")
    .agg(F.count(F.lit(1)).cast("long").alias("pickup_count"))
)

print("Agrégats zone-heure avec au moins un pickup :", df_pickups.count())
display(df_pickups.orderBy("taxi_zone_id", "hour_local").limit(10))
print("Vérification du nombre total de pickups :")
assert df_pickups.agg(F.sum("pickup_count")).first()[0] == n_taxi, "Total des pickups différent du total Taxi Silver"

## 4 — Créer la grille exhaustive zone × heure
Les heures avec zéro trajet ne figurent pas dans `df_pickups`. Sans grille exhaustive, le modèle ne verrait jamais les périodes de demande nulle.

Pour 2025 et 5 zones : **5 × 365 × 24 = 43 800 lignes nominales**. Il s'agit d'une grille d'heures **locales naïves** (restriction DST signalée plus haut).

In [ ]:
df_zones = spark.createDataFrame([(int(z),) for z in ZONE_IDS], ["taxi_zone_id"])

df_hours = spark.sql("""
SELECT explode(sequence(
  to_timestamp_ntz('2025-01-01 00:00:00'),
  to_timestamp_ntz('2025-12-31 23:00:00'),
  INTERVAL 1 HOUR
)) AS hour_local
""")

df_grid = df_zones.crossJoin(F.broadcast(df_hours))
assert df_grid.count() == EXPECTED_ROWS, "Grille zone-heure incomplète"

print("Grille :", EXPECTED_ROWS, "lignes")
display(df_grid.orderBy("taxi_zone_id", "hour_local").limit(5))

## 5 — Ajouter les pickups, y compris les heures à zéro
La jointure est `left` depuis la grille complète. L'absence de trajet est transformée en `pickup_count = 0` (**uniquement après l'agrégation**).

In [ ]:
df_demand = (
    df_grid
    .join(df_pickups, ["taxi_zone_id", "hour_local"], "left")
    .withColumn("pickup_count", F.coalesce(F.col("pickup_count"), F.lit(0)).cast("long"))
)

print("Heures à zéro pickup :", df_demand.filter(F.col("pickup_count") == 0).count())
print("Somme des pickups :", df_demand.agg(F.sum("pickup_count")).first()[0])
assert df_demand.count() == EXPECTED_ROWS
assert df_demand.agg(F.sum("pickup_count")).first()[0] == n_taxi
assert df_demand.filter(F.col("pickup_count") < 0).limit(1).count() == 0

## 6 — Vérifier la couverture météo avant la jointure
La météo doit avoir **une ligne par zone + heure** et couvrir **chaque** clé de la grille. On vérifie les clés manquantes et les éventuels enregistrements hors grille. Pas de remplacement silencieux des observations météo manquantes.

In [ ]:
df_weather_keys = df_weather.select(
    "taxi_zone_id", F.col("weather_hour_local").alias("hour_local")
)

missing_weather = df_grid.join(df_weather_keys, ["taxi_zone_id", "hour_local"], "left_anti")
extra_weather = df_weather_keys.join(df_grid, ["taxi_zone_id", "hour_local"], "left_anti")
n_missing = missing_weather.count()
n_extra = extra_weather.count()
print("Heures météo manquantes :", n_missing)
print("Heures météo hors grille :", n_extra)
if n_missing:
    display(missing_weather.orderBy("taxi_zone_id", "hour_local").limit(20))
if n_extra:
    display(extra_weather.orderBy("taxi_zone_id", "hour_local").limit(20))
assert n_missing == 0 and n_extra == 0, (
    "Couverture temporelle météo incompatible. Vérifier les changements d'heure DST, "
    "les données source et les timestamps avant de poursuivre."
)

## 7 — Jointure Taxi × Météo
La jointure utilise exclusivement `(taxi_zone_id, hour_local)`. Chaque heure/zone doit rester unique. Les variables météo sont préfixées **`weather_`** pour clarifier leur provenance. `source_file` et `zone_slug` restent des métadonnées descriptives et ne seront pas utilisés comme variables ML.

In [ ]:
WEATHER_FEATURES = [
    "temperature_2m", "relative_humidity_2m", "precipitation", "rain",
    "snowfall", "wind_speed_10m", "weather_code"
]
missing_cols = set(WEATHER_FEATURES) - set(df_weather.columns)
assert not missing_cols, f"Variables météo manquantes : {missing_cols}"

weather_select = df_weather.select(
    "taxi_zone_id",
    F.col("weather_hour_local").alias("hour_local"),
    *[F.col(c).alias("weather_" + c) for c in WEATHER_FEATURES]
)

df_joined = df_demand.join(weather_select, ["taxi_zone_id", "hour_local"], "inner")
assert df_joined.count() == EXPECTED_ROWS, "Jointure météo a perdu/dupliqué des heures"
assert df_joined.agg(F.sum("pickup_count")).first()[0] == n_taxi
print("Jointure effectuée :", EXPECTED_ROWS, "lignes")
display(df_joined.orderBy("taxi_zone_id", "hour_local").limit(5))

## 8 — Feature engineering : calendrier et cycles
Toutes les caractéristiques ci-dessous sont connues à l'avance lors d'une prédiction future. **Convention :** `day_of_week` suit Spark : dimanche=1, samedi=7. Aucune variable dérivée de la demande future n'est utilisée.

In [ ]:
df_gold = (
    df_joined
    .withColumn("year", F.year("hour_local"))
    .withColumn("month", F.month("hour_local"))
    .withColumn("day_of_month", F.dayofmonth("hour_local"))
    .withColumn("day_of_week", F.dayofweek("hour_local"))
    .withColumn("hour_of_day", F.hour("hour_local"))
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("hour_sin", F.sin(F.lit(2 * math.pi) * F.col("hour_of_day") / F.lit(24.0)))
    .withColumn("hour_cos", F.cos(F.lit(2 * math.pi) * F.col("hour_of_day") / F.lit(24.0)))
    .withColumn("dow_sin", F.sin(F.lit(2 * math.pi) * (F.col("day_of_week") - 1) / F.lit(7.0)))
    .withColumn("dow_cos", F.cos(F.lit(2 * math.pi) * (F.col("day_of_week") - 1) / F.lit(7.0)))
    .withColumn("month_sin", F.sin(F.lit(2 * math.pi) * (F.col("month") - 1) / F.lit(12.0)))
    .withColumn("month_cos", F.cos(F.lit(2 * math.pi) * (F.col("month") - 1) / F.lit(12.0)))
)

print("Schéma Gold :")
df_gold.printSchema()
display(df_gold.orderBy("taxi_zone_id", "hour_local").limit(10))

## 9 — Contrôles finaux et exploration Gold
Vérifier : total des lignes, une observation par zone/heure, absence de NULL ou NaN, absence de valeurs négatives de cible, intégrité de la somme des pickups et répartition par zone/mois. La couverture météo complète est testée plus haut.

In [ ]:
gold_count = df_gold.count()
assert gold_count == EXPECTED_ROWS, f"Gold contient {gold_count} lignes, attendu {EXPECTED_ROWS}"
assert df_gold.select("taxi_zone_id", "hour_local").distinct().count() == gold_count, "Doublons de clé Gold"
assert df_gold.agg(F.sum("pickup_count")).first()[0] == n_taxi, "Perte/duplication de pickups"
assert df_gold.filter(F.col("pickup_count") < 0).limit(1).count() == 0

nulls = df_gold.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_gold.columns
]).first().asDict()
assert all(v == 0 for v in nulls.values()), f"NULL dans Gold : { {k:v for k,v in nulls.items() if v} }"

float_cols = [f.name for f in df_gold.schema.fields if isinstance(f.dataType, (T.FloatType, T.DoubleType))]
if float_cols:
    nans = df_gold.select([F.sum(F.isnan(F.col(c)).cast("int")).alias(c) for c in float_cols]).first().asDict()
    assert all(v == 0 for v in nans.values()), f"NaN dans Gold : {nans}"

print("=== Statistiques de la cible par zone ===")
display(df_gold.groupBy("taxi_zone_id").agg(
    F.count("*").alias("heures"),
    F.sum("pickup_count").alias("total_pickups"),
    F.avg("pickup_count").alias("moyenne_horaire"),
    F.max("pickup_count").alias("pic_horaire"),
    F.sum(F.when(F.col("pickup_count") == 0, 1).otherwise(0)).alias("heures_zero")
).orderBy("taxi_zone_id"))

print("=== Totaux mensuels ===")
display(df_gold.groupBy("month").agg(F.sum("pickup_count").alias("total_pickups"))
        .orderBy("month"))
print("Validations Gold réussies.")

## 10 — Écrire la table Delta Gold (action explicite)
**N'exécuter qu'après avoir vérifié les contrôles précédents.** `overwrite` remplace uniquement `gold.taxi_demand_features` (pas les tables Silver). Schémas Lakehouse requis.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print("Table créée :", GOLD_TABLE)
spark.sql(f"DESCRIBE DETAIL {GOLD_TABLE}").select("format", "location").show(truncate=False)
print("Total lignes persistées :", spark.table(GOLD_TABLE).count())

## 11 — Préparer le contrat ML et la séparation chronologique (sans entraîner)
La cible est `pickup_count`. Le tableau Gold contient les variables calendaires et météo qui devront être **produites à l'identique** lors de l'inférence. Le modèle final devra être évalué sur des périodes postérieures à l'entraînement.

**Attention** : les observations météo 2025 issues de l'API *historical forecast* ne constituent pas automatiquement des prévisions « connues à l'époque ». Pour une évaluation opérationnelle stricte, il faudra aligner les données météo utilisées à la date de leur émission (prévisions *as-of*). On peut déjà réaliser une évaluation rétrospective, en mentionnant cette limite.

Le prochain notebook traitera **XGBoost + MLflow**. Aucun lag calculé à partir de la cible n'est introduit ici.

In [ ]:
ML_FEATURES = [
    "taxi_zone_id", "month", "day_of_month", "day_of_week", "hour_of_day", "is_weekend",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "weather_temperature_2m", "weather_relative_humidity_2m",
    "weather_precipitation", "weather_rain", "weather_snowfall",
    "weather_wind_speed_10m", "weather_weather_code"
]
TARGET = "pickup_count"

# Découpage temporel de référence (adaptable dans le notebook ML)
# Train : janv.-août ; validation : sept.-oct. ; test : nov.-déc.
df_ml_source = spark.table(GOLD_TABLE)
df_ml_split = df_ml_source.withColumn(
    "dataset_split",
    F.when(F.col("hour_local") < F.lit("2025-09-01"), F.lit("train"))
     .when(F.col("hour_local") < F.lit("2025-11-01"), F.lit("validation"))
     .otherwise(F.lit("test"))
)

print("Variables ML (hors cible) :", ML_FEATURES)
print("Variable cible :", TARGET)
display(df_ml_split.groupBy("dataset_split").count().orderBy("dataset_split"))
# Note : les données du split restent virtuelles, rien de nouveau n'est écrit ici.